# Exercise 4 — Train ResNet on Kaggle GPU

This notebook clones the repo, sets up the data, runs the tests, trains the solar-cell defect classifier, and exports an ONNX model for the leaderboard.

## Step 0 — Verify the GPU
Confirms PyTorch can see a CUDA GPU. If `cuda available: False`, stop and enable the GPU accelerator in Settings — training on CPU here is impractically slow.

In [ ]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
!nvidia-smi -L

## Step 1 — Clone the repository
Pulls a fresh copy of the repo into Kaggle's writable `/kaggle/working` directory and moves into the exercise folder. `--depth 1` fetches only the latest commit (faster, smaller), and the `rm -rf` first makes re-runs start clean. Requires **Internet → On** and your latest code pushed to GitHub.

In [ ]:
%cd /kaggle/working
!rm -rf deep-learning
!git clone --depth 1 https://github.com/mo-karbalaee/deep-learning.git
%cd /kaggle/working/deep-learning/ex4/src_to_implement

## Step 2 — Install dependencies
Kaggle images already ship torch, torchvision, scikit-image, pandas and scikit-learn. We only add `onnxruntime` (runs the ONNX part of the test suite) and `onnxscript` (required by torch 2.x's ONNX exporter). The import line fails fast if anything essential is missing.

In [ ]:
!pip -q install onnxruntime onnxscript
import skimage, pandas, sklearn, torchvision
print('deps OK')

## Step 3 — Extract the dataset
`images.zip` is committed in the repo; this unzips it into `images/` (~2000 electroluminescence PNGs) next to `data.csv`, which is where `ChallengeDataset` expects to find them. It is skipped automatically if the folder already exists.

In [ ]:
import zipfile, os
if not os.path.isdir('images'):
    zipfile.ZipFile('images.zip').extractall('.')
print(len(os.listdir('images')), 'images extracted')

## Step 4 — Run the unit tests (optional)
Runs the graded suite with the `Bonus` flag: `TestDataset` (data pipeline shape + normalization) and `TestModel` (forward pass + ONNX export/reload). This verifies *code correctness* only — an untrained model passes. Safe to skip if you just want to train.

In [ ]:
!python PytorchChallengeTests.py Bonus

## Step 5 — Train the model
Runs the full pipeline on the GPU: 80/20 train/val split, `BCELoss` (multi-label, since a cell can be both cracked and inactive), Adam, and early stopping (patience 10). It prints validation loss and mean F1 each epoch and writes to `checkpoints/` only when the validation loss improves. This is the slow, compute-heavy step. Tune `epochs` / `batch_size` / `lr` inside `train.py` if the F1 plateaus below the 0.60 pass threshold.

In [ ]:
!python train.py

## Step 6 — Export the best model to ONNX
Selects the best checkpoint (the highest-numbered one, since checkpoints are saved only on improvement), exports it to ONNX via `export_onnx.py`, and copies the `.onnx` (plus the loss curve) into `/kaggle/working/`. Download it from the **Output** tab and upload it to the leaderboard.

In [ ]:
import glob, os, re, shutil
ckpts = sorted(glob.glob('checkpoints/checkpoint_*.ckp'))
assert ckpts, 'No checkpoints found - did training run?'
best = ckpts[-1]
epoch = int(re.findall(r'(\d+)', os.path.basename(best))[0])
print('best checkpoint:', best, '| epoch', epoch)
!python export_onnx.py {epoch}
onnx_name = 'checkpoint_{:03d}.onnx'.format(epoch)
shutil.copy(onnx_name, '/kaggle/working/' + onnx_name)
if os.path.exists('losses.png'):
    shutil.copy('losses.png', '/kaggle/working/losses.png')
print('Saved /kaggle/working/' + onnx_name)